In [36]:
from ollama import Client
import os
from dotenv import load_dotenv
import json


In [37]:
load_dotenv()
ollama_api_key = os.getenv('OLLAMA_API_KEY')
ollama_api_key

'b934c867b190485ea55d0ab35c950db8.VlUlCD4gyYycsnVrAS7O8W7B'

In [38]:
headers = {
    'Authorization': f'Bearer {ollama_api_key}'
}

In [39]:
fake_events ={
        'hyderabad':'No new events for today',
        'bangalore':'Election campaign is going on',
        'chennai':'2 Ranji matches are played today',
        'goa':'fashion events planned '
    }

def get_latest_events_based_on_city(city:str)->str:
    ''' get events based on city'''

    if city.lower() in fake_events.keys:
        return fake_events[city.lower()]
    else:
        return f'no information available for this city : {city}'
    

    

In [40]:
tools = [
    {
        'type':'function',
        'function':{
            'name':'get_latest_events_based_on_city',
            'description': 'Get latest events based on city provided by user',
            'parameters':{
                'type':'object',
                'properties':{
                    'city':{
                        'type':'strig',
                        'description':'name of the city, eg: Hyderabad',
                    }
                },
                'required':['city']
            }
        }
    }
]

available_tools = {
    "get_latest_events_based_on_city": get_latest_events_based_on_city,
}

In [41]:
def get_messages(prompt):
    messages = [
        {
            'role':'assistant',
            'content':'Answer the user question in 2 lines'
        },
        {
            'role':'user',
            'content':prompt
        }
    ]
    return messages


In [42]:
client = Client(host = 'https://ollama.com',
                headers = headers)

def ollama_chat(prompt, messages):
    response = client.chat(
        model = 'gemma4:31b-cloud',
        messages=messages, 
        tools = tools
    )
    print(f'response : {response}')
    print(f'response : {response['message']['content']}')

    if response['message']['tool_calls'] is not None:
        tool_call = response['message']['tool_calls'][0]
        function_name = tool_call.function.name
        print(f'toll_call : {tool_call}')
        arguments = json.loads(tool_call.function.arguments)
        print(f'arguments - {arguments}')
        # print(f'calling {function_name} with {arguments}')
        # result = available_tools[function_name](**arguments)

In [44]:
prompt = 'start asking'
count = 0
while prompt.lower() != 'quit' and count < 3:
    prompt = input('Ask Question or enter quit to stop session ')
    if prompt.lower() != 'quit':
        messages = get_messages(prompt)
        ollama_chat(prompt ,messages)
        count = count + 1
    else:
        print(f'quiting session')
    

response : model='gemma4:31b-cloud' created_at='2026-06-16T06:34:29.154141887Z' done=True done_reason='stop' total_duration=332692493 load_duration=None prompt_eval_count=103 prompt_eval_duration=None eval_count=23 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_latest_events_based_on_city', arguments={'city': 'Hyderabad'}))]) logprobs=None
response : 
toll_call : function=Function(name='get_latest_events_based_on_city', arguments={'city': 'Hyderabad'})


TypeError: the JSON object must be str, bytes or bytearray, not dict